# 30 -- The reference study's split: zone 13 held out, no buffer

The reference study validates at Tuktoyaktuk on one contiguous zone. Its
patching notebook sorts every patch by x offset, splits the sorted list into
10 equal-count bins, and labels them `region_id` 10..19; `config.yaml`
holds out zone **13** (the 4th bin from the west, ~168 patches). Patch ids
were assigned sequentially in bin order from 12000, so zone 13 should be
roughly ids 12504-12671 -- printed below as a sanity check. No buffer is
applied and every other patch is used for training; patches that straddle
the zone edge therefore overlap validation patches, and the notebook counts
how many.

Everything else is the leading configuration on the final pipeline
(`repeat` channels, real attributes, k = 3, standardised from this training
split's statistics, 100 epochs, PLMS). Result goes beside the reference's
Tuktoyaktuk value (ZNCC 0.740 PLMS / 0.670 DDIM) in `tab:benchmark`, so the
sensor is compared under the reference's own split as well as under ours.

| tag | validation | training | seed |
|---|---|---|---|
| `std_realattrs_refsplit` | zone 13 (~168) | all others, no buffer (~1500) | 42 |


In [ ]:
import os, sys, json, random, time, datetime as dt
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import rasterio

assert torch.cuda.is_available(), 'CUDA is required.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIRS = {'IW': WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc',
           'EW': WORKING_REPO / 'input_data' / 's1_patches_tuk_ew'}
LIDAR_SURVEY_DATE = dt.date(2024, 4, 16)

TARGET_HW = (256, 256); BATCH_SIZE = 8; EPOCHS = 100; TIMESTEPS = 1000; LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15; SPLIT_SEED = 42; NOISE_SCHEDULE = 'linear'; ATTENTION_VARIANT = 'default'
BLOCK_SIZE_M, BUFFER_M = 1024.0, 150.0
NUM_WORKERS = 4
EVAL_SAMPLER_NAME = 'plms'

CONFIGS = [
    dict(tag='std_realattrs_refsplit', channels='repeat', attrs='real', k=3, data='IW', seed=42),
]
N_ZONES, HELD_OUT_ZONE = 10, 13   # reference study: 10 x-balanced bins labelled 10..19, zone 13 held out
def ckpt_path(tag): return CHECKPOINT_DIR / f's1_tuk_pcrtc_{tag}_unet_best.pth'
def metrics_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_{EVAL_SAMPLER_NAME}_validation_metrics.json'
def history_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_history.json'
def stats_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_norm_stats.json'

for d in S1_DIRS.values(): assert d.exists(), f'missing {d}'

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc
import inspect
assert 'cond_channels_per_view' in inspect.signature(ConditionalUNet.__init__).parameters, \
    'tessa_baseline ConditionalUNet lacks cond_channels_per_view -- apply the workstation patch from 15/16 first'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

EVAL_SAMPLER = {'plms': p_sample_loop_plms, 'ddim': p_sample_loop_ddim}[EVAL_SAMPLER_NAME]
print('evaluation sampler:', EVAL_SAMPLER_NAME)

## Data (as `26`), plus the reference study's zone split

In [ ]:
from scipy.ndimage import median_filter

def load_sar_db(time_path, despeckle=0):
    with rasterio.open(time_path) as src:
        sar = src.read()[:2].astype(np.float32)
    sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
    if despeckle:
        sar = median_filter(sar, size=(1, despeckle, despeckle))   # per band, on linear power, at native resolution, before dB
    return 10.0 * np.log10(np.maximum(sar, 1e-12))

def build_real_attrs(s1_path, times, context_k):
    attrs_list = json.load(open(s1_path / 'attrs.json')) if (s1_path / 'attrs.json').exists() else []
    vecs = []
    for t in times:
        idx = int(t.stem[1:]); a = attrs_list[idx] if idx < len(attrs_list) else {}
        age = (dt.date.fromisoformat(a['acquisition_date']) - LIDAR_SURVEY_DATE).days / 30.0 if a.get('acquisition_date') else 0.0
        vecs.append([age, 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0, (a.get('relative_orbit_number') or 0) / 175.0, 0, 0, 0, 0, 0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()

class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k, channels, attrs_mode, norm_stats, despeckle=0):
        self.s1_dir, self.lidar_dir, self.patch_ids = Path(s1_dir), Path(lidar_dir), list(patch_ids)
        self.context_k, self.channels, self.attrs_mode, self.norm_stats, self.despeckle = context_k, channels, attrs_mode, norm_stats, despeckle
    def __len__(self): return len(self.patch_ids)
    def __getitem__(self, i):
        pid = self.patch_ids[i]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{pid}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]; mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        pm = float(target[mask].sum() / max(1, int(mask.sum()))); target = (target - pm) * mask
        s1_path = self.s1_dir / f's1_patch_{pid}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k: raise RuntimeError(f'{s1_path} has fewer than {self.context_k} views')
        views = []
        mean, std = self.norm_stats
        for t in times:
            sar = (load_sar_db(t, self.despeckle) - mean[:, None, None]) / std[:, None, None]
            st = F.interpolate(torch.from_numpy(sar).unsqueeze(0), size=TARGET_HW, mode='bilinear', align_corners=False).squeeze(0)
            views.append(st.repeat(2, 1, 1) if self.channels == 'repeat' else st)
        cond = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k) if self.attrs_mode == 'real' else torch.zeros(8 * self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': cond.float(),
                'attrs': attrs, 'patch_mean': torch.tensor(pm), 'patch_id': pid}

def spatial_split(s1_dir):
    lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
    s1_ids = {p.name.split('_')[-1] for p in Path(s1_dir).glob('s1_patch_*') if p.is_dir()}
    paired = sorted(lidar_ids & s1_ids)
    blocks, dropped = {}, []
    for pid in paired:
        with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src: b = src.bounds
        cx, cy = (b.left + b.right) / 2, (b.bottom + b.top) / 2
        bx, by = int(cx // BLOCK_SIZE_M), int(cy // BLOCK_SIZE_M)
        d = min(cx - bx * BLOCK_SIZE_M, (bx + 1) * BLOCK_SIZE_M - cx, cy - by * BLOCK_SIZE_M, (by + 1) * BLOCK_SIZE_M - cy)
        (dropped if d < BUFFER_M else blocks.setdefault((bx, by), [])).append(pid)
    ids = list(blocks); random.Random(SPLIT_SEED).shuffle(ids)
    target_val = int(len(paired) * VAL_FRACTION); val, train, n = [], [], 0
    for bid in ids:
        if n < target_val: val.extend(blocks[bid]); n += len(blocks[bid])
        else: train.extend(blocks[bid])
    assert not (set(train) & set(val))
    return train, val, len(dropped)

def compute_stats(s1_dir, train_ids, k, despeckle=0):
    sums = np.zeros(2); sqs = np.zeros(2); count = 0
    for pid in train_ids:
        for t in sorted((Path(s1_dir) / f's1_patch_{pid}').glob('t*.tif'))[:k]:
            sar = load_sar_db(t, despeckle); sums += sar.reshape(2, -1).sum(1); sqs += (sar.reshape(2, -1) ** 2).sum(1); count += sar.shape[1] * sar.shape[2]
    mean = sums / count; std = np.sqrt(np.maximum(sqs / count - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

def bounds_of(pid):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src: b = src.bounds
    return (b.left, b.bottom, b.right, b.top)

def edge_distance(a, b):
    dx = max(b[0] - a[2], a[0] - b[2], 0.0); dy = max(b[1] - a[3], a[1] - b[3], 0.0)
    return (dx * dx + dy * dy) ** 0.5

def reference_zone_split(s1_dir):
    """Reference study's rule: sort paired patches by x (left edge), split into N_ZONES equal-count bins,
    hold out bin (HELD_OUT_ZONE - 10). No buffer; all other patches train."""
    lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
    s1_ids = {p.name.split('_')[-1] for p in Path(s1_dir).glob('s1_patch_*') if p.is_dir()}
    paired = sorted(lidar_ids & s1_ids)
    global BOUNDS
    BOUNDS = {pid: bounds_of(pid) for pid in paired}
    order = sorted(paired, key=lambda p: (BOUNDS[p][0], BOUNDS[p][1]))   # x then y, as np.argsort on x offsets
    bins = np.array_split(np.array(order), N_ZONES)
    val = sorted(bins[HELD_OUT_ZONE - 10].tolist()); vs = set(val)
    train = [p for p in paired if p not in vs]
    return train, val, 0

SPLITS = {'IW': reference_zone_split(S1_DIRS['IW'])}
tr, va, _ = SPLITS['IW']
ids = sorted(int(p) for p in va)
print(f'reference zone split: train={len(tr)} val={len(va)} (zone {HELD_OUT_ZONE}); val ids {ids[0]}..{ids[-1]}, '
      f'{sum(1 for a, b in zip(ids, ids[1:]) if b == a + 1)} consecutive pairs of {len(ids)-1}  (expect ~12504..12671 if ids follow bin order)')
x_lo = min(BOUNDS[p][0] for p in va); x_hi = max(BOUNDS[p][2] for p in va)
print(f'zone {HELD_OUT_ZONE} spans x = {x_lo:.0f}..{x_hi:.0f} m ({x_hi - x_lo:.0f} m wide)')

# how leaky is the reference split?  (recorded for the caveat, not asserted)
from shapely.geometry import box
from shapely.strtree import STRtree
tboxes = [box(*BOUNDS[p]) for p in tr]; tree = STRtree(tboxes)
n_overlap = sum(1 for v in va if any(tboxes[k].intersects(box(*BOUNDS[v])) and not tboxes[k].touches(box(*BOUNDS[v])) for k in tree.query(box(*BOUNDS[v]))))
min_sep = min(min(edge_distance(BOUNDS[v], BOUNDS[t]) for t in tr) for v in va)
print(f'{n_overlap} / {len(va)} validation patches overlap at least one training patch (no buffer); min separation {min_sep:.1f} m')
json.dump({'train': tr, 'val': va, 'zone': HELD_OUT_ZONE, 'n_val_overlapping_train': n_overlap, 'min_sep_m': min_sep,
           'rule': f'{N_ZONES} x-balanced bins, zone {HELD_OUT_ZONE} held out, no buffer (reference study)'},
          open(OUTPUT_DIR / 's1_pcrtc_refsplit_split.json', 'w'), indent=1)


## Train + evaluate one configuration (as `29`; statistics from this training split)

In [ ]:
def masked_mse(pred, target, mask):
    valid = mask.bool().unsqueeze(1); return ((pred - target) ** 2)[valid].mean()

def run_config(cfg):
    tag, k = cfg['tag'], cfg['k']
    epochs = cfg.get('epochs', EPOCHS)
    if metrics_path(tag).exists():
        print(f'[{tag}] metrics exist -- skipping'); return json.load(open(metrics_path(tag)))
    s1_dir = S1_DIRS[cfg['data']]; train_ids, val_ids, _ = SPLITS[cfg['data']]
    despeckle = cfg.get('despeckle', 0)
    # the training split differs from 19's, so the standardisation statistics are recomputed from it (training ids only)
    stats = compute_stats(s1_dir, train_ids, k, despeckle)
    json.dump({'mean': stats[0].tolist(), 'std': stats[1].tolist(), 'n_train': len(train_ids), 'k': k, 'despeckle': despeckle}, open(stats_path(tag), 'w'), indent=2)
    print(f'[{tag}] train={len(train_ids)} val={len(val_ids)}  stats VV {stats[0][0]:.2f}/{stats[1][0]:.2f}  VH {stats[0][1]:.2f}/{stats[1][1]:.2f}')

    seed_everything(cfg['seed'])
    ds_tr = LidarS1Dataset(s1_dir, LIDAR_DIR, train_ids, k, cfg['channels'], cfg['attrs'], stats, despeckle)
    ds_va = LidarS1Dataset(s1_dir, LIDAR_DIR, val_ids,   k, cfg['channels'], cfg['attrs'], stats, despeckle)
    train_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    cpv = 4 if cfg['channels'] == 'repeat' else 2
    model = ConditionalUNet(in_channels=1, cond_channels=cpv * k, attr_dim=8 * k, base_channels=128, embed_dim=256, unet_depth=4,
                            attention_variant=ATTENTION_VARIANT, cond_k=k, cond_channels_per_view=cpv).to(DEVICE)
    scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scaler = GradScaler()

    history = {'train_loss': [], 'val_loss': []}; best = float('inf'); t0 = time.time()
    for epoch in range(epochs):
        model.train(); tr_tot = 0.0
        for b in train_loader:
            target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
            ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); tr_tot += loss.item()
        model.eval(); va_tot = 0.0
        with torch.no_grad():
            for b in val_loader:
                target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
                ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
                with autocast():
                    va_tot += masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask).item()
        tr, va = tr_tot / max(1, len(train_loader)), va_tot / max(1, len(val_loader))
        history['train_loss'].append(tr); history['val_loss'].append(va)
        if (epoch + 1) % 10 == 0 or epoch == 0: print(f'[{tag}] epoch {epoch+1:03d}/{epochs} train={tr:.6f} val={va:.6f}  {(time.time()-t0)/60:.0f} min')
        if va < best:
            best = va
            torch.save({'model_state_dict': model.state_dict(), 'config': {**cfg, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'standardised': True},
                        'epoch': epoch + 1, 'val_loss': va}, ckpt_path(tag))
    json.dump(history, open(history_path(tag), 'w'))

    # evaluate best checkpoint with EVAL_SAMPLER (PLMS, the reference study's choice; 24 showed +0.05..0.07 over DDIM)
    model.load_state_dict(torch.load(ckpt_path(tag), map_location=DEVICE)['model_state_dict']); model.eval()
    seed_everything(SPLIT_SEED); rows = []
    with torch.no_grad():
        for b in val_loader:
            target, cond, attrs = b['lidar'].to(DEVICE), b['s1'].to(DEVICE), b['attrs'].to(DEVICE)
            mask = b['mask'].to(DEVICE).bool(); pm = b['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
            pred = EVAL_SAMPLER(model, scheduler, target.shape, cond, attrs, DEVICE)
            gt_abs, pred_abs = target + pm, pred + pm
            for i, pid in enumerate(b['patch_id']):
                g, p, m = gt_abs[i], pred_abs[i], mask[i]
                gv, pv = g.squeeze()[m].cpu().numpy(), p.squeeze()[m].cpu().numpy()
                rows.append({'patch_id': pid, 'rmse_m': float(rmse(g, p, m)), 'bias_m': float(bias(g, p, m)),
                             'sigma_error_pct': float(sigma_error(g, p, m)),
                             'normal_angle_error_deg': float(normal_angle_error(g, p, m, pixel_size=1.0, degrees=True)),
                             'jsd': float(average_jsd_multiscale(g, p, pixel_size=1.0, mask=m)),
                             'psd_rmse': float(log_psd_rmse(g, p, pixel_size=1.0, mask=m)), 'zncc': float(zncc(g, p, m)),
                             'gt_std_val': float(gv.std()), 'pred_std_val': float(pv.std())})
    json.dump(rows, open(metrics_path(tag), 'w'), indent=2)
    mean = {key: float(np.nanmean([r[key] for r in rows])) for key in rows[0] if key != 'patch_id'}
    print(f'[{tag}] DONE  ZNCC {mean["zncc"]:+.4f}  RMSE {mean["rmse_m"]:.4f}  sig% {mean["sigma_error_pct"]:.1f}  pred/gt {mean["pred_std_val"]:.4f}/{mean["gt_std_val"]:.4f}  ({(time.time()-t0)/3600:.1f} h)')
    del model, optimizer; torch.cuda.empty_cache()
    return rows

## Run

In [ ]:
results = {}
for cfg in CONFIGS:
    try:
        results[cfg['tag']] = run_config(cfg)
    except Exception as exc:
        print(f'[{cfg["tag"]}] FAILED: {type(exc).__name__}: {exc}')
        raise

## Summary against the reference study's Tuktoyaktuk value and our blocked split

In [ ]:
def mean_of(path):
    if not Path(path).exists(): return None
    rows = json.load(open(path)); return {k: float(np.nanmean([r[k] for r in rows])) for k in rows[0] if k != 'patch_id'}
rows = [
    ('optical, reference split, PLMS (ref. study)', dict(zncc=0.7398, rmse_m=0.1025, sigma_error_pct=18.4, jsd=0.0667, psd_rmse=1.627, pred_std_val=float('nan'))),
    ('optical, reference split, DDIM (ref. study)', dict(zncc=0.6701, rmse_m=0.1159, sigma_error_pct=15.0, jsd=0.0604, psd_rmse=1.404, pred_std_val=float('nan'))),
    ('radar, reference split, PLMS (this run)',    mean_of(metrics_path('std_realattrs_refsplit'))),
    ('radar, blocked split, PLMS (27)',            mean_of(OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_standardised_plms_validation_metrics.json')),
    ('radar, val-buffer split, PLMS (29)',         mean_of(OUTPUT_DIR / 's1_pcrtc_std_realattrs_valbuf_plms_validation_metrics.json')),
]
print(f'{"run":<46}{"ZNCC":>8}{"RMSE":>8}{"sig%":>7}{"JSD":>8}{"PSD":>8}{"pred std":>9}')
print('-' * 94)
for label, m in rows:
    if m is None: print(f'{label:<46}{"--":>8}'); continue
    print(f'{label:<46}{m["zncc"]:>8.4f}{m["rmse_m"]:>8.4f}{m["sigma_error_pct"]:>7.1f}{m["jsd"]:>8.4f}{m["psd_rmse"]:>8.4f}{m["pred_std_val"]:>9.4f}')
print('\nNote: the optical rows are the reference study\'s per-region patch summaries for Tuktoyaktuk (zone 13, 168 patches);'
      ' its model was trained on Pond Inlet + Tuktoyaktuk with six Sentinel-2 views.')
